# US County COVID-19 Surge Risk Prediction

**Question:** Can recent reported COVID-19 trends help flag US counties at risk of a rise in reported cases over the next seven days?

This notebook creates a historical county-level early-warning classifier. A surge is defined as next-week reported cases per 100,000 being at least **25% higher** than the preceding 7-day total and at least **10 per 100,000**. These are project-defined thresholds, not official public-health thresholds.

**Important:** This is retrospective analysis of historical reported data, not a live warning system. Reported cases are affected by testing and reporting practices; predictions do not establish causation.

## 1. Setup and input paths

Place `usa_county_wise(1).csv` and `time_series_covid_19_deaths_US.csv` in `data/raw/` beside this notebook, or change the paths below.

In [ ]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, ConfusionMatrixDisplay,
    classification_report, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
SURGE_GROWTH = 1.25
MIN_FUTURE_CASES_PER_100K = 10.0
HORIZON_DAYS = 7
COUNTY_FILE = Path('data/raw/usa_county_wise(1).csv')
POPULATION_FILE = Path('data/raw/time_series_covid_19_deaths_US.csv')

print('County file exists:', COUNTY_FILE.exists(), COUNTY_FILE)
print('Population file exists:', POPULATION_FILE.exists(), POPULATION_FILE)
# If files are elsewhere, update COUNTY_FILE and POPULATION_FILE to their full paths.

## 2. Load and inspect the datasets

In [ ]:
if not COUNTY_FILE.exists() or not POPULATION_FILE.exists():
    raise FileNotFoundError('Put both CSVs in data/raw/ or update the paths in the setup cell.')

county_raw = pd.read_csv(COUNTY_FILE, low_memory=False)
population_raw = pd.read_csv(POPULATION_FILE, low_memory=False)
print('County shape:', county_raw.shape)
print('Population shape:', population_raw.shape)
display(county_raw.head())
print('County columns:', county_raw.columns.tolist())
print('Population columns:', population_raw.columns.tolist())

## 3. Clean and join population data

County and state names are normalized before joining. Counties without valid population values are excluded because population-adjusted incidence cannot be calculated for them.

In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ''
    value = str(value).lower().strip()
    value = re.sub(r'\\s+', ' ', value)
    value = re.sub(r'\\s+(county|parish|borough|census area|municipality)$', '', value)
    return value

required = {'Admin2', 'Province_State', 'Date', 'Confirmed'}
if required - set(county_raw.columns):
    raise ValueError(f'County file missing columns: {sorted(required - set(county_raw.columns))}')
if not {'Admin2','Province_State','Population'}.issubset(population_raw.columns):
    raise ValueError('Population file needs Admin2, Province_State and Population columns.')

county = county_raw.copy()
county['Date'] = pd.to_datetime(county['Date'], errors='coerce')
county['Confirmed'] = pd.to_numeric(county['Confirmed'], errors='coerce')
county['Deaths'] = pd.to_numeric(county['Deaths'], errors='coerce') if 'Deaths' in county else np.nan
if 'Country_Region' in county:
    county = county[county['Country_Region'].astype(str).str.upper().isin(['US','USA','UNITED STATES'])]
county = county.dropna(subset=['Date','Confirmed','Admin2','Province_State']).copy()
county['state_key'] = county['Province_State'].map(normalize_name)
county['county_key'] = county['Admin2'].map(normalize_name)

pop = population_raw[['Province_State','Admin2','Population']].copy()
pop['state_key'] = pop['Province_State'].map(normalize_name)
pop['county_key'] = pop['Admin2'].map(normalize_name)
pop['Population'] = pd.to_numeric(pop['Population'], errors='coerce')
pop = pop.dropna(subset=['Population'])
pop = pop[pop['Population'] > 0].drop_duplicates(['state_key','county_key'])
pop = pop[['state_key','county_key','Population']]

county = county.merge(pop, on=['state_key','county_key'], how='left')
county = county[county['Population'].notna() & (county['Population'] > 0)]
county = county[~county['county_key'].isin(['','unassigned','unknown'])]
county = county.groupby(['state_key','county_key','Province_State','Admin2','Date'], as_index=False).agg(
    Confirmed=('Confirmed','max'), Deaths=('Deaths','max'), Population=('Population','first'))
county = county.sort_values(['state_key','county_key','Date']).reset_index(drop=True)
print(f'Rows retained: {len(county):,}')
print(f'Counties retained: {county[["state_key","county_key"]].drop_duplicates().shape[0]:,}')
print('Date range:', county.Date.min().date(), 'to', county.Date.max().date())

## 4. Build features and the surge target

Daily increments are calculated from cumulative counts. Negative increments from retrospective corrections are clipped to zero. Features use information available on or before the prediction date; the future 7-day window is used only to construct the label.

In [ ]:
county['daily_cases'] = county.groupby(['state_key','county_key'])['Confirmed'].diff().fillna(0).clip(lower=0)
county['daily_deaths'] = county.groupby(['state_key','county_key'])['Deaths'].diff().fillna(0).clip(lower=0)
county['cases_per_100k'] = county['daily_cases'] / county['Population'] * 100_000
county['deaths_per_100k'] = county['daily_deaths'] / county['Population'] * 100_000

frames = []
for _, g in county.groupby(['state_key','county_key'], sort=False):
    g = g.sort_values('Date').copy().reset_index(drop=True)
    inc = g['cases_per_100k'].astype(float)
    death = g['deaths_per_100k'].astype(float)
    g['incidence_7d'] = inc.rolling(7, min_periods=7).sum()
    g['incidence_14d'] = inc.rolling(14, min_periods=14).sum()
    g['incidence_28d'] = inc.rolling(28, min_periods=28).sum()
    g['prior_7d_incidence'] = inc.shift(7).rolling(7, min_periods=7).sum()
    g['incidence_growth_7d'] = (g['incidence_7d'] - g['prior_7d_incidence']) / (g['prior_7d_incidence'] + 1.0)
    g['daily_incidence_mean_7d'] = inc.rolling(7, min_periods=7).mean()
    g['daily_incidence_std_7d'] = inc.rolling(7, min_periods=7).std()
    g['deaths_7d_per_100k'] = death.rolling(7, min_periods=7).sum()
    g['day_of_week'] = g['Date'].dt.dayofweek
    g['month'] = g['Date'].dt.month
    future = pd.concat([inc.shift(-i) for i in range(1, 8)], axis=1)
    g['future_7d_incidence'] = future.sum(axis=1, min_count=7)
    valid = g['incidence_7d'].notna() & g['future_7d_incidence'].notna()
    g['surge'] = np.nan
    g.loc[valid, 'surge'] = ((g.loc[valid,'future_7d_incidence'] >= SURGE_GROWTH*g.loc[valid,'incidence_7d']) &
                             (g.loc[valid,'future_7d_incidence'] >= MIN_FUTURE_CASES_PER_100K)).astype(int)
    # One prediction origin per county per week reduces near-duplicate samples.
    frames.append(g.iloc[::7].copy())

model_data = pd.concat(frames, ignore_index=True).dropna(subset=['surge']).copy()
model_data['surge'] = model_data['surge'].astype(int)
model_data = model_data.replace([np.inf,-np.inf], np.nan)
print('Model rows:', f'{len(model_data):,}')
print('Surge label rate:', f'{model_data.surge.mean():.1%}')
display(model_data[['Date','Province_State','Admin2','incidence_7d','future_7d_incidence','surge']].head())

## 5. Chronological train/test split

The latest 20% of unique dates are held out. This is preferable to a random row split for a forecasting-style task.

In [ ]:
dates = np.sort(model_data['Date'].unique())
if len(dates) < 60:
    raise ValueError('Not enough unique dates for a useful chronological holdout.')
test_start = pd.Timestamp(dates[int(len(dates)*0.80)])
train = model_data[model_data['Date'] < test_start].copy()
test = model_data[model_data['Date'] >= test_start].copy()
print('Train:', len(train), '| Test:', len(test))
print('Train ends:', train.Date.max().date())
print('Test period:', test.Date.min().date(), 'to', test.Date.max().date())
print('Train surge rate:', f'{train.surge.mean():.1%}', '| Test surge rate:', f'{test.surge.mean():.1%}')
if train.surge.nunique() < 2 or test.surge.nunique() < 2:
    warnings.warn('One split contains only one target class; some metrics may be undefined.')

## 6. Train and compare models

Models are compared using precision, recall, F1, average precision and ROC-AUC. Average precision is useful when surge labels are imbalanced. The 0.50 probability cutoff is a starting point, not an official risk threshold.

In [ ]:
numeric = ['incidence_7d','incidence_14d','incidence_28d','prior_7d_incidence','incidence_growth_7d',
           'daily_incidence_mean_7d','daily_incidence_std_7d','deaths_7d_per_100k','day_of_week','month']
categorical = ['Province_State']
features = numeric + categorical
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
models = {
    'Logistic Regression': (Pipeline([('prep', preprocessor),('model', LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))]), features),
    'Random Forest': (Pipeline([('prep', preprocessor),('model', RandomForestClassifier(n_estimators=250,max_depth=14,min_samples_leaf=8,class_weight='balanced_subsample',random_state=42,n_jobs=-1))]), features),
    'HistGradientBoosting': (Pipeline([('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(max_iter=180,learning_rate=0.07,max_leaf_nodes=15,l2_regularization=1.0,random_state=42))]), numeric)
}
X_train, X_test = train[features], test[features]
y_train, y_test = train['surge'].astype(int), test['surge'].astype(int)
results, fitted, prediction_tables = [], {}, []
for name, (model, cols) in models.items():
    print('Training', name)
    model.fit(train[cols], y_train)
    prob = model.predict_proba(test[cols])[:,1]
    pred = (prob >= 0.5).astype(int)
    results.append({'model':name,
        'precision':precision_score(y_test,pred,zero_division=0),
        'recall':recall_score(y_test,pred,zero_division=0),
        'f1':f1_score(y_test,pred,zero_division=0),
        'average_precision':average_precision_score(y_test,prob) if y_test.nunique()>1 else np.nan,
        'roc_auc':roc_auc_score(y_test,prob) if y_test.nunique()>1 else np.nan})
    fitted[name] = model
    prediction_tables.append(pd.DataFrame({'Date':test.Date.to_numpy(),'Province_State':test.Province_State.to_numpy(),
        'Admin2':test.Admin2.to_numpy(),'Population':test.Population.to_numpy(),
        'observed_surge':y_test.to_numpy(),'predicted_surge':pred,'surge_probability':prob,
        'recent_7d_cases_per_100k':test.incidence_7d.to_numpy(),
        'future_7d_cases_per_100k':test.future_7d_incidence.to_numpy(),'model':name}))

metrics = pd.DataFrame(results).sort_values(['average_precision','f1'],ascending=False).reset_index(drop=True)
display(metrics.style.format({c:'{:.3f}' for c in ['precision','recall','f1','average_precision','roc_auc']}))

## 7. Evaluate the best model

In [ ]:
predictions = pd.concat(prediction_tables, ignore_index=True)
best_name = metrics.iloc[0]['model']
best = predictions[predictions.model == best_name].copy()
print('Selected model:', best_name)
print(classification_report(best['observed_surge'],best['predicted_surge'],target_names=['No surge','Surge'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(best['observed_surge'],best['predicted_surge'],
    display_labels=['No surge','Surge'],cmap='Blues',colorbar=False)
plt.title(f'County surge classification — {best_name}')
plt.tight_layout(); plt.show()

## 8. Inspect county risk scores

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for label, subset in best.groupby('observed_surge'):
    ax.hist(subset['surge_probability'],bins=25,alpha=0.6,label='Observed surge' if label else 'No observed surge')
ax.set_xlabel('Predicted surge probability'); ax.set_ylabel('County-week count')
ax.set_title(f'Risk score distribution — {best_name}'); ax.legend(); plt.tight_layout(); plt.show()

latest_date = best.Date.max()
latest = best[best.Date == latest_date].sort_values('surge_probability',ascending=False)
print('Latest test date:', pd.Timestamp(latest_date).date())
display(latest[['Province_State','Admin2','surge_probability','observed_surge',
                'recent_7d_cases_per_100k','future_7d_cases_per_100k']].head(20))

## 9. Export files for GitHub and Power BI

The CSVs can be imported into Power BI. The model binary is saved locally for reproducibility.

In [ ]:
from pathlib import Path
import joblib
output_dir = Path('outputs'); output_dir.mkdir(exist_ok=True)
metrics.to_csv(output_dir/'model_metrics.csv',index=False)
predictions.to_csv(output_dir/'county_surge_predictions.csv',index=False)
model_data[['Date','Province_State','Admin2','Population','incidence_7d','incidence_14d','incidence_28d',
            'incidence_growth_7d','deaths_7d_per_100k','future_7d_incidence','surge']].to_csv(
                output_dir/'county_weekly_features.csv',index=False)
joblib.dump(fitted[best_name],output_dir/'best_surge_model.joblib')
print('Saved outputs to:', output_dir.resolve())
print('Files:', sorted(p.name for p in output_dir.iterdir()))

## 10. How to report the findings

When you run this notebook, write your conclusion using the actual metric values generated above. Report the test period, selected model, precision, recall, F1 and average precision. Discuss false alarms versus missed surges.

### Limitations
- This is retrospective prediction, not current outbreak surveillance.
- Reported cases are not the same as all infections.
- Negative daily changes from data revisions were clipped to zero.
- The surge definition is project-specific and should be sensitivity-tested.
- Weekly prediction origins can still have partially overlapping windows.
- Population and county-name matching may introduce errors.
- Predictive relationships do not establish causation.